In [ ]:
# This script generates the following tables:
# - Pollen Counts per slide
# - Pollen counts per trap (summed over slides)
# - Percentages, considering only 14 classes (= the 13 target pollen taxa and "Other" (including non-tagret taxa), and without Lycopodium, Non-Pollen and indeterminates (blurry or covered)

# The table for pollen counts per slide used in the associated publication is also available in the ZENODO repository "4.Tables_for_figures/comptages_brutauto_all_perslide.csv"

In [2]:
import pandas as pd
import os
import pickle
import numpy as np

In [3]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent))
from config1 import MONITORING_DATA, classes18, fig_class_order

In [4]:
mp_pred_combined = MONITORING_DATA / "PREDICTIONS_51slides_combined/"
p_save_table  = MONITORING_DATA / "Tables_PollenCounts/"

In [5]:
rows = []
for slide_k in os.listdir(mp_pred_combined):
    slide_name=slide_k.replace('combinedpred_', '').replace('.pkl', '')
    with open(mp_pred_combined/slide_k, "rb") as f:
        d_slide_k = pickle.load(f)
    site, year = slide_name.split("_")[0], slide_name.split("_")[1]
    row = {"site": site,"year": year,"slide": slide_name}
    
    sum_vector = np.sum(list(d_slide_k.values()), axis=0)

    row.update(dict(zip(classes18, sum_vector)))

    rows.append(row)

df = pd.DataFrame(rows)
df.head()

,site,year,slide,QuercusDeciduous,QuercusIlex,Buxus,Phillyrea,Fraxinus,Olea,Cupressaceae,...,Poaceae,Plantago,VitisF,VitisS,Pinaceae,Other,IndetBlurry,IndetCovered,NonPollen,Lycopodium
0,D2,2022,D2_2022_A,1.517178,2.767463,1.500652,0.194405,0.170016,2.391095,10.402325,...,2.391928,1.326727,0.656717,0.103561,16.992980,29.784357,46.307455,14.702064,170.655278,175.563158
1,W3,2019,W3_2019_A,136.760719,478.391169,2177.727851,138.312435,100.824449,15.345525,185.896101,...,23.107383,7.500566,22.346689,24.269021,110.711749,243.195644,358.872052,451.050119,308.613608,55.919316
2,W1,2019,W1_2019_A,75.359585,181.637652,50.587787,945.935313,218.442994,40.706964,90.886663,...,20.188402,4.985038,39.362655,7.941687,89.998735,121.984964,134.142815,417.727126,200.322475,17.819520
3,W1,2020,W1_2020_B,32.040627,83.158429,29.916726,281.672557,249.281187,90.220296,88.544227,...,35.605694,10.791352,154.623657,5.097808,130.522791,526.969602,93.604467,148.843252,411.229137,167.513537
4,W2,2023,W2_2023_A,107.298160,638.544638,9.540890,8.414192,8.088298,29.188295,139.257198,...,38.159829,36.665519,133.811162,20.972936,163.302955,1166.268517,326.038441,653.704480,1140.137518,376.250332


In [6]:
id_cols = ["site", "year",  "slide"]



df_long = df.melt(
    id_vars=id_cols,
    var_name="class",
    value_name="count"
)

df_long["class_num"] = [classes18.index(el) for el in df_long["class"]]
df_long=df_long.sort_values(by=["class_num", "site", "year"])
df_long["site_year"] = (
    df_long["site"].astype(str) + "_" + df_long["year"].astype(str)
)

df_long

,site,year,slide,class,count,class_num,site_year
25,D1,2019,D1_2019_A,QuercusDeciduous,29.147301,0,D1_2019
9,D1,2020,D1_2020_A,QuercusDeciduous,25.037277,0,D1_2020
12,D1,2020,D1_2020_B,QuercusDeciduous,88.580480,0,D1_2020
26,D1,2021,D1_2021_A,QuercusDeciduous,78.337110,0,D1_2021
6,D2,2020,D2_2020_B,QuercusDeciduous,49.359799,0,D2_2020
...,...,...,...,...,...,...,...
914,W4,2022,W4_2022_B,Lycopodium,415.785535,17,W4_2022
916,W4,2022,W4_2022_C,Lycopodium,681.618849,17,W4_2022
904,W4,2023,W4_2023_A,Lycopodium,685.774463,17,W4_2023
905,W4,2023,W4_2023_B,Lycopodium,534.484342,17,W4_2023


In [ ]:
#df_long.to_csv(p_save_table / "comptages_brutauto_all_perslide.csv", index=False) 


### counts per slide are also available in the repository :

In [6]:
df_long = pd.read_csv(MONITORING_DATA / "Tables_for_figures" / "comptages_brutauto_all_perslide.csv")
df_long

,site,year,slide,class,count,class_num,site_year
0,D1,2019,D1_2019_A,QuercusDeciduous,29.147301,0,D1_2019
1,D1,2020,D1_2020_A,QuercusDeciduous,25.037277,0,D1_2020
2,D1,2020,D1_2020_B,QuercusDeciduous,88.580480,0,D1_2020
3,D1,2021,D1_2021_A,QuercusDeciduous,78.337110,0,D1_2021
4,D2,2020,D2_2020_B,QuercusDeciduous,49.359799,0,D2_2020
...,...,...,...,...,...,...,...
913,W4,2022,W4_2022_B,Lycopodium,415.785535,17,W4_2022
914,W4,2022,W4_2022_C,Lycopodium,681.618849,17,W4_2022
915,W4,2023,W4_2023_A,Lycopodium,685.774463,17,W4_2023
916,W4,2023,W4_2023_B,Lycopodium,534.484342,17,W4_2023


In [8]:
print(sum(df_long["count"])) # 281518, OK !

281517.9999999996


In [9]:
df_long_2 = df_long.drop(columns="slide")
df_long_2

,site,year,class,count,class_num,site_year
25,D1,2019,QuercusDeciduous,29.147301,0,D1_2019
9,D1,2020,QuercusDeciduous,25.037277,0,D1_2020
12,D1,2020,QuercusDeciduous,88.580480,0,D1_2020
26,D1,2021,QuercusDeciduous,78.337110,0,D1_2021
6,D2,2020,QuercusDeciduous,49.359799,0,D2_2020
...,...,...,...,...,...,...
914,W4,2022,Lycopodium,415.785535,17,W4_2022
916,W4,2022,Lycopodium,681.618849,17,W4_2022
904,W4,2023,Lycopodium,685.774463,17,W4_2023
905,W4,2023,Lycopodium,534.484342,17,W4_2023


In [10]:
# pollen counts per traps
df_all_sum = df_long_2.groupby(["class", "site", "year", "site_year", 'class_num']).sum().reset_index() 

# df_all_sum.to_csv(p_save_table  / "comptages_brutauto_all_pertrap.csv", index=False)

df_all_sum.head()


,class,site,year,site_year,class_num,count
0,Buxus,D1,2019,D1_2019,2,53.806745
1,Buxus,D1,2020,D1_2020,2,41.644678
2,Buxus,D1,2021,D1_2021,2,24.990120
3,Buxus,D2,2020,D2_2020,2,27.706941
4,Buxus,D2,2021,D2_2021,2,30.323475


## percentages

In [11]:

df_siteyear_perc = df_all_sum[df_all_sum["class"].isin(fig_class_order)]

df_siteyear_perc.loc[:, "proportion"] = df_siteyear_perc["count"] * 100 / df_siteyear_perc.groupby(["site", "year"])["count"].transform("sum")

df_siteyear_perc

,class,site,year,site_year,class_num,count,proportion
0,Buxus,D1,2019,D1_2019,2,53.806745,6.734485
1,Buxus,D1,2020,D1_2020,2,41.644678,1.226623
2,Buxus,D1,2021,D1_2021,2,24.990120,1.016147
3,Buxus,D2,2020,D2_2020,2,27.706941,0.795045
4,Buxus,D2,2021,D2_2021,2,30.323475,0.712326
...,...,...,...,...,...,...,...
499,VitisS,W4,2019,W4_2019,11,82.988265,3.651658
500,VitisS,W4,2020,W4_2020,11,160.987076,3.775502
501,VitisS,W4,2021,W4_2021,11,166.132552,2.248256
502,VitisS,W4,2022,W4_2022,11,363.848496,2.812331


In [12]:
# saving 
# df_siteyear_perc.to_csv(p_save_table / "percentages_all.csv", index=False)



In [ ]:
# end